# SCORING AND VISUALISATION

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2
import hippo
import mrich
from mrich import print
from pathlib import Path
from os import environ
import pandas as pd
import plotly.express as px

## Reinitialise Directories for Current Cycle

In [ ]:
target_name = "Target name"  # Change this to match the target name in Fragalysis.
cycle_number = 1  # Change this to match the design cycle you are scoring.

In [ ]:
# all XChem-FFF work is kept under $HOME2/XChem-FFF; each target gets its own
# subdirectory here, and each design cycle its own subdirectory within that
xchem_fff_dir = Path(environ["HOME2"]) / "XChem-FFF"
target_dir = xchem_fff_dir / target_name.lower()
cycle_dir = target_dir / f"cycle_{cycle_number:01}"
gnina_dir = cycle_dir / "gnina"
gnina_outputs_dir = gnina_dir / "outputs"
bulk_targets_dir = Path(environ["BULK"]) / "TARGETS"
bulk_target_dir = bulk_targets_dir / target_name

## Animal

In [ ]:
animal = hippo.HIPPO(target_name, bulk_target_dir / f"{target_name}.sqlite")

In [ ]:
animal.tags  # Display available tags

## 1. Build a score table for the GNINA-minimised poses

In [ ]:
method_tags = ["fragmenstein", "pure_knitwork", "impure_knitwork"]

In [ ]:
data = []

for method in method_tags:
    poses = animal.poses.get_by_tag(method)
    for pose in poses:
        data.append({
            "pose_id": pose.id,
            "compound_id": pose.compound_id,
            "method": method,
            "energy_score": pose.energy_score,
            "distance_score": pose.distance_score,
        })

score_df = pd.DataFrame(data).set_index("pose_id")
score_df.head()

In [ ]:
score_df.to_csv(gnina_dir / f"{target_name}_c{cycle_number:02}_scores.csv")

## 2. Visualise the score distributions

In [ ]:
px.histogram(score_df, x="energy_score", color="method", barmode="overlay")

In [ ]:
px.scatter(score_df, x="distance_score", y="energy_score", color="method", hover_data=["compound_id"])

## 3. Select the top-scoring poses

In [ ]:
score_quantile = 0.2  # keep the best-scoring 20% of poses for each method

In [ ]:
cutoffs = score_df.groupby("method")["energy_score"].quantile(score_quantile)

top_score_df = score_df[
    score_df.apply(lambda row: row["energy_score"] <= cutoffs[row["method"]], axis=1)
]
top_score_df

In [ ]:
top_poses = animal.poses[list(top_score_df.index)]
top_poses

## 4. Generate pose overlays and images of the top poses

In [ ]:
top_poses.interactive()  # 3D overlay of the top-scoring poses in the binding site

In [ ]:
for pose in top_poses[:5]:
    print(pose)
    pose.draw()  # 2D depiction of the pose's compound

## 5. Export the top-scoring poses

In [ ]:
top_score_df.to_csv(gnina_dir / f"{target_name}_c{cycle_number:02}_top_scoring_poses.csv")

In [ ]:
top_poses.to_fragalysis(
    str(gnina_dir / f"{target_name}_c{cycle_number:02}_top_scoring_poses.sdf"),
    method = "gnina",
    submitter_name = "YOUR NAME HERE",
    submitter_email = "YOUR EMAIL HERE",
    submitter_institution = "YOUR INSTITUTION HERE",
    copy_reference_pdbs=True
)

In [ ]:
animal.db.backup()  # save the updated HIPPO database